# ML-03 — Frame Your Lane as an ML Task

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/shreyamT/flyrank_new/blob/main/work/notebooks/w02_ml_task_framing.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. My lane as an ML task (type)

*Classification, clustering, ranking, or scoring — which one, and why?*

## 2. Target or proxy

*What would you predict? Where does that label come from — observed outcome or a defined rule?*

`is_declining_label = 1` when `trend_direction == "down"`, meaning impressions in the last 30 days fell more than 20% against the previous 30 days. This is a **proxy** for "this page needs a refresh". `trend_direction` and `trend_pct` must never be used as features, because they define the label (leakage).


## 3. Success metric

*One metric you can defend. What number means 'good'?*

**Precision@50 on a client-holdout split** (a client's pages never appear in both train and test). About 54% of pages are "down", so random picking scores about 0.54, and I compare against that and against a hand-written rule.

## 4. The unit of analysis, as a real dataframe

*Load your lane's slice and show it: one row = one what?*

In [1]:
import pandas as pd
from pathlib import Path

# Find the starter CSV from wherever the notebook is running
for p in [Path("data/raw/content_refresh_anonymized.csv"),
          Path("../data/raw/content_refresh_anonymized.csv"),
          Path("../../data/raw/content_refresh_anonymized.csv")]:
    if p.exists():
        data_path = p
        break
else:
    raise FileNotFoundError("Run this from inside the repo")

df = pd.read_csv(data_path)
df["is_declining_label"] = (df["trend_direction"] == "down").astype(int)

# One row = one page (content item), with 90-day totals
cols = ["content_id", "client_id", "content_type", "avg_position",
        "impressions_90d", "impressions_prev_30d", "impressions_last_30d",
        "trend_pct", "trend_direction", "is_declining_label"]
display(df[cols].head(5))

print(f"Rows: {len(df):,} | Clients: {df['client_id'].nunique()}")
print("\nTarget column (is_declining_label) share:")
print(df["is_declining_label"].value_counts(normalize=True).round(3))

,content_id,client_id,content_type,avg_position,impressions_90d,impressions_prev_30d,impressions_last_30d,trend_pct,trend_direction,is_declining_label
0,content_304f48230142,client_f369cb89fc,keyword article,10.6,3803,987,578,-41.4,down,1
1,content_a1fb4e703a9e,client_4e07408562,keyword article,20.3,15320,5915,2501,-57.7,down,1
2,content_9aa793d4d895,client_7f2253d7e2,keyword article,36.5,12581,6089,2382,-60.9,down,1
3,content_331d6c4de07b,client_19581e27de,keyword article,6.2,11751,4206,3626,-13.8,stable,0
4,content_d99b7a2d90ca,client_3fdba35f04,keyword article,44.0,19140,6452,4211,-34.7,down,1


Rows: 30,000 | Clients: 32

Target column (is_declining_label) share:
is_declining_label
1    0.542
0    0.458
Name: proportion, dtype: float64


## 5. Why ML beats a fixed rule here

*What makes the pattern too messy for an if-statement?*

**Why ML beats a fixed rule here (or doesn't).**
In my Week 1 run, the starter pipeline's hand-written rule scored Precision@50 = 0.24 on client holdout, while the random forest scored 0.74. Picking pages at random would score about 0.54, because 54.2% of pages are "down". So the rule was below random and the model was about 20 points above it.
ML earns its place here because many weak signals (position, impressions, content age, CTR) have to be combined, and a one-line rule can only use one or two of them.
Caveat: that rule was a weak baseline. A sharper hand rule ("stale and visible") scored close to a small decision tree in my Week 2 run, so what I observed is that ML beats a weak rule, not necessarily a good one.
The label and the 90-day totals overlap in time, so this detects pages that are already declining and does not forecast. I would only trust the model after testing it on unseen clients against a sharper hand rule, and after rebuilding it so the features come from before the label window.
**Action:** editors review the top 25 to 50 ranked pages each week and refresh them (decision-support, not an automatic decision).

## Self-check

Before you submit, confirm each line honestly:

- [x] Every section above is filled — markdown thinking AND the code that backs it
- [x] The notebook runs top to bottom with no errors (Runtime → Run all)
- [x] No client names, URLs, or private queries anywhere
- [x] My claims use careful words: observed, measured, directional, decision-support
- [x] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.